# MODULE 1 — Fine-tune 3 mô hình NLI (Evidence Quantification Engine)

So sánh 3 ứng viên cho tầng "Custom Cross-Encoder NLI" (Tầng 2 trong Tri-Layer
Hybrid Engine) của module Evidence Quantification, trên cùng 1 dataset 3 nhãn
(`supported` / `contradicted` / `insufficient`) sinh ra bởi
`scripts/finetune_nli/01_generate_dataset.py` từ chính kho PDF thật của dự án.

**3 ứng viên và lý do chọn (2 trục so sánh, không phải 3 model ngẫu nhiên):**

| # | Model | Base weights | Trục so sánh |
|---|---|---|---|
| A | `microsoft/deberta-v3-xsmall` | Chỉ pretrain MLM, train NLI từ đầu | Baseline nhỏ nhất |
| B | `microsoft/deberta-v3-small` | Chỉ pretrain MLM, train NLI từ đầu | So với A: **kích thước có giúp ích không?** |
| C | `cross-encoder/nli-deberta-v3-xsmall` | Đã pretrain sẵn trên SNLI+MultiNLI | So với A: **transfer learning từ NLI có giúp ích không?** (cùng kích thước với A) |

**Hướng dẫn thao tác:**
1. Mở [Google Colab](https://colab.research.google.com/) → `Runtime` → `Change runtime type` → chọn **T4 GPU**.
2. Chạy Cell 1 để cài thư viện.
3. Chạy Cell 2, khi được hỏi thì upload 3 file: `train.jsonl`, `val.jsonl`, `test.jsonl`
   (nằm ở `scripts/finetune_nli/data/` sau khi chạy `01_generate_dataset.py` ở máy local).
4. Chạy tuần tự các cell còn lại — quá trình train cả 3 model mất khoảng **15-30 phút** trên T4
   (tuỳ kích thước dataset), KHÔNG mất 30-45 phút/model như ước tính ban đầu vì cả 3
   model đều rất nhỏ.
5. Cell cuối cùng sẽ in bảng so sánh + tự động nén và tải xuống (`download()`) model
   thắng cuộc — giải nén vào `models/nli_evidence_v1/` trong repo rồi commit.


In [ ]:
# Cell 1: Cài đặt thư viện
# sentencepiece bat buoc cho tokenizer DeBERTa-v3 -- thieu no se crash voi loi
# tiktoken BPE parse khi load AutoTokenizer (da xac nhan that khi smoke-test cuc bo
# tren may local).
#
# KHONG upgrade torch: Colab da co san torch+torchvision ban CUDA-matched. Upgrade
# rieng torch (khong upgrade torchvision cung luc) lam lech phien ban ABI giua 2
# thu vien -- torchvision._meta_registrations mat op "torchvision::nms", roi
# transformers.trainer_utils lazy-import peft (co san tren Colab) -> peft co gang
# import BloomPreTrainedModel -> keo theo torchvision -> crash ngay tu cau lenh
# `from transformers import TrainingArguments` o Cell 4. Da xac nhan loi that nay
# khi chay tren Colab. Chi upgrade cac goi Python thuan (khong co compiled
# extension phai khop ABI voi torch).
!pip install -q --upgrade transformers datasets accelerate evaluate scikit-learn sentencepiece
print("OK - đã cài xong (giữ nguyên torch/torchvision có sẵn của Colab).")

In [ ]:
# Cell 2: Upload 3 file dataset (train.jsonl / val.jsonl / test.jsonl)
import os

IN_COLAB = "google.colab" in str(get_ipython())

if IN_COLAB:
    from google.colab import files
    print("Chọn và upload CẢ 3 file: train.jsonl, val.jsonl, test.jsonl")
    uploaded = files.upload()
    for fname in uploaded:
        print("  ->", fname)
else:
    # Chạy local để debug nhanh (không có GPU, chỉ để test pipeline không lỗi cú pháp)
    print("Không phát hiện môi trường Colab -- giả định 3 file jsonl đã có sẵn trong thư mục hiện tại.")

for required in ("train.jsonl", "val.jsonl", "test.jsonl"):
    assert os.path.exists(required), f"Thiếu file {required} -- upload lại Cell 2."
print("OK - đủ 3 file.")

In [ ]:
# Cell 3: Nạp dataset + định nghĩa nhãn
import json
from datasets import Dataset, DatasetDict

LABEL2ID = {"contradicted": 0, "insufficient": 1, "supported": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            r = json.loads(line)
            rows.append({"premise": r["premise"], "hypothesis": r["hypothesis"], "label": LABEL2ID[r["label"]]})
    return rows

raw = DatasetDict({
    "train": Dataset.from_list(load_jsonl("train.jsonl")),
    "validation": Dataset.from_list(load_jsonl("val.jsonl")),
    "test": Dataset.from_list(load_jsonl("test.jsonl")),
})

print(raw)
print("\nPhân bố nhãn (train):")
from collections import Counter
print(Counter(raw["train"]["label"]))

In [ ]:
# Cell 4: Hàm dùng chung để train + đánh giá 1 model
import time
import os
import shutil
import inspect
import numpy as np
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback,
)
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị train: {DEVICE}")


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )
    return {"accuracy": acc, "f1_macro": f1, "precision_macro": precision, "recall_macro": recall}


def build_training_args(**kwargs):
    supported = set(inspect.signature(TrainingArguments.__init__).parameters.keys())
    filtered = {k: v for k, v in kwargs.items() if k in supported}
    dropped = {k: v for k, v in kwargs.items() if k not in supported}
    if dropped:
        print(f"  [CANH BAO] TrainingArguments ban nay khong ho tro: {list(dropped.keys())} -- da bo qua.")
    return TrainingArguments(**filtered)


def train_and_eval(model_name: str, run_name: str, output_dir: str, epochs: int = 6, lr: float = 1e-5):
    print("\n" + "=" * 80)
    print(f"BAT DAU: {run_name}  ({model_name})")
    print("=" * 80)

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tokenize(batch):
        return tokenizer(batch["premise"], batch["hypothesis"], truncation=True, max_length=384, padding="max_length")

    tokenized = raw.map(tokenize, batched=True)
    tokenized = tokenized.remove_columns(["premise", "hypothesis"])
    tokenized.set_format("torch")

    # torch_dtype=torch.float32 co chu dich, KHONG duoc bo qua: mot so checkpoint
    # goc tren HF Hub (vi du microsoft/deberta-v3-xsmall va -small) duoc publish
    # san o fp16 de tiet kiem bang thong. Neu khong ep kieu tuong minh o day,
    # from_pretrained() giu nguyen fp16 xuyen suot qua trinh train (fp16=False
    # trong TrainingArguments chi tat AMP autocast luc TINH TOAN, KHONG doi dtype
    # goc cua trong so), va model se duoc luu lai o fp16. Suy luan fp16 tren CPU
    # (khong co Tensor Core nhu GPU) cham hon fp32 hang chuc lan -- da xac nhan
    # that: model A/B (tu checkpoint Microsoft) bi dtype float16 mot cach vo tinh,
    # latency CPU do duoc la ~2.2-2.3 GIAY/cau, trong khi model C (tu checkpoint
    # cross-encoder, von da o fp32) chi ~130-200ms/cau -- CUNG kien truc, CUNG
    # kich thuoc, chi khac dtype. Sau khi ep torch_dtype=float32 o day thi ca 3
    # model deu train/luu o fp32, so sanh cong bang.
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID,
        ignore_mismatched_sizes=True, torch_dtype=torch.float32,
    )

    n_params = sum(p.numel() for p in model.parameters())

    batch_size = 16
    steps_per_epoch = max(1, len(tokenized["train"]) // batch_size)
    total_steps = steps_per_epoch * epochs
    warmup_steps = max(1, int(total_steps * 0.1))

    args = build_training_args(
        output_dir=f"./_runs/{run_name}",
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=32,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_steps=warmup_steps,
        adam_epsilon=1e-6,
        max_grad_norm=1.0,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        fp16=False,
        logging_steps=10,
        report_to=[],
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model, args=args,
        train_dataset=tokenized["train"], eval_dataset=tokenized["validation"],
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    t0 = time.time()
    trainer.train()
    train_seconds = time.time() - t0

    # Đánh giá thật trên TEST set (chưa từng thấy trong lúc train/chọn checkpoint)
    test_output = trainer.predict(tokenized["test"])
    test_preds = np.argmax(test_output.predictions, axis=-1)
    test_labels = test_output.label_ids

    test_acc = accuracy_score(test_labels, test_preds)
    p, r, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="macro", zero_division=0)
    cm = confusion_matrix(test_labels, test_preds, labels=[0, 1, 2]).tolist()
    report_str = classification_report(
        test_labels, test_preds, labels=[0, 1, 2],
        target_names=[ID2LABEL[i] for i in [0, 1, 2]], zero_division=0,
    )

    # Lưu model
    os.makedirs(output_dir, exist_ok=True)
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    size_mb = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, fs in os.walk(output_dir) for f in fs
    ) / (1024 * 1024)

    # Đo độ trễ suy luận TRÊN CPU (đúng môi trường EC2 sẽ chạy thật, không phải GPU)
    cpu_model = AutoModelForSequenceClassification.from_pretrained(output_dir).to("cpu").eval()
    cpu_tokenizer = AutoTokenizer.from_pretrained(output_dir)
    sample = tokenized["test"][:20]
    latencies = []
    with torch.no_grad():
        for i in range(20):
            text_pair = (raw["test"][i]["premise"], raw["test"][i]["hypothesis"])
            enc = cpu_tokenizer(*text_pair, truncation=True, max_length=384, return_tensors="pt")
            t0 = time.perf_counter()
            cpu_model(**enc)
            latencies.append((time.perf_counter() - t0) * 1000)
    avg_latency_ms = float(np.mean(latencies))
    p95_latency_ms = float(np.percentile(latencies, 95))

    result = {
        "run_name": run_name,
        "model_name": model_name,
        "n_params": int(n_params),
        "train_seconds": round(train_seconds, 1),
        "model_size_mb": round(size_mb, 1),
        "cpu_latency_ms_avg": round(avg_latency_ms, 2),
        "cpu_latency_ms_p95": round(p95_latency_ms, 2),
        "test_accuracy": round(float(test_acc), 4),
        "test_f1_macro": round(float(f1), 4),
        "test_precision_macro": round(float(p), 4),
        "test_recall_macro": round(float(r), 4),
        "confusion_matrix": cm,
        "confusion_matrix_labels": ["contradicted", "insufficient", "supported"],
        "classification_report": report_str,
    }

    print(f"\nKET QUA {run_name}:")
    print(f"  Tham so: {n_params/1e6:.1f}M | Kich thuoc dia: {size_mb:.1f} MB | Thoi gian train: {train_seconds:.1f}s")
    print(f"  Test Accuracy: {test_acc*100:.2f}% | Test F1-macro: {f1*100:.2f}%")
    print(f"  CPU latency: {avg_latency_ms:.2f} ms/cap (p95: {p95_latency_ms:.2f} ms)")
    print(report_str)

    del model, cpu_model
    torch.cuda.empty_cache() if DEVICE == "cuda" else None

    return result

In [ ]:
# Cell 5: Train + danh gia ca 3 model (~15-30 phut tren T4)
CANDIDATES = [
    ("microsoft/deberta-v3-xsmall",        "A_xsmall_from_scratch", "./models_out/A_xsmall_from_scratch"),
    ("microsoft/deberta-v3-small",         "B_small_from_scratch",  "./models_out/B_small_from_scratch"),
    ("cross-encoder/nli-deberta-v3-xsmall","C_xsmall_nli_pretrained","./models_out/C_xsmall_nli_pretrained"),
]

all_results = []
for model_name, run_name, out_dir in CANDIDATES:
    res = train_and_eval(model_name, run_name, out_dir)
    all_results.append(res)

print("\n\nDA TRAIN XONG CA 3 MODEL.")

In [ ]:
# Cell 6: Bang so sanh tong hop (so lieu that cho bao cao) + chon model thang cuoc
import pandas as pd
import json

df = pd.DataFrame([{
    "Model": r["run_name"],
    "Base checkpoint": r["model_name"],
    "Tham so": f"{r['n_params']/1e6:.1f}M",
    "Kich thuoc (MB)": r["model_size_mb"],
    "Thoi gian train (s)": r["train_seconds"],
    "Test Accuracy": f"{r['test_accuracy']*100:.2f}%",
    "Test F1-macro": f"{r['test_f1_macro']*100:.2f}%",
    "CPU latency avg (ms)": r["cpu_latency_ms_avg"],
    "CPU latency p95 (ms)": r["cpu_latency_ms_p95"],
} for r in all_results])

print("=" * 100)
print("BANG SO SANH 3 MODEL — MODULE 1 EVIDENCE QUANTIFICATION ENGINE")
print("=" * 100)
print(df.to_string(index=False))

# Quy tac chon winner: uu tien F1-macro (chat luong phan loai la quan trong nhat cho
# 1 he thong phat hien hallucination), chi dung latency/size de phan xu khi F1-macro
# cach nhau duoi 1.5 diem % (trong sai so ngau nhien cua 1 lan train tren dataset nho).
best = max(all_results, key=lambda r: r["test_f1_macro"])
close_contenders = [r for r in all_results if best["test_f1_macro"] - r["test_f1_macro"] <= 0.015]
if len(close_contenders) > 1:
    winner = min(close_contenders, key=lambda r: r["cpu_latency_ms_avg"])
    print(f"\n>> {len(close_contenders)} model co F1-macro sat nhau (trong 1.5%) -> chon theo CPU latency thap nhat.")
else:
    winner = best

print(f"\n*** MODEL DUOC CHON: {winner['run_name']} ({winner['model_name']}) ***")
print(f"    Test F1-macro: {winner['test_f1_macro']*100:.2f}% | CPU latency: {winner['cpu_latency_ms_avg']:.2f} ms | Kich thuoc: {winner['model_size_mb']:.1f} MB")

with open("comparison_report.json", "w", encoding="utf-8") as f:
    json.dump({"results": all_results, "winner": winner["run_name"]}, f, indent=2, ensure_ascii=False)

df.to_csv("comparison_report.csv", index=False)
print("\nDa luu comparison_report.json va comparison_report.csv")

In [ ]:
# Cell 7: Nen model thang cuoc va tai xuong
import shutil

winner_dir = [c[2] for c in CANDIDATES if c[1] == winner["run_name"]][0]
zip_path = shutil.make_archive("nli_evidence_v1_winner", "zip", winner_dir)
print(f"Da nen: {zip_path}")

if IN_COLAB:
    from google.colab import files
    files.download(zip_path)
    files.download("comparison_report.json")
    files.download("comparison_report.csv")
    print("\nSau khi tai ve: giai nen vao models/nli_evidence_v1/ trong repo, roi chay:")
    print("  python scripts/finetune_nli/03_compare_models.py --report comparison_report.json")
else:
    print(f"Model thang cuoc dang nam o: {winner_dir}")
    print(f"File nen: {zip_path}")